# Data S4 FAFB connectomics Investigation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Initial Exploration of FAFB neurons used

In [ ]:
s4_df = pd.read_csv("../data/DataS4_FAFBreconstruction.csv")
s4_df.head()

In [ ]:
all_columns = s4_df.columns.tolist()
print("All columns in the DataFrame:")
for col in all_columns:
    print(f"    {col}")

Ok so here's the quick breakdown of the columns and their meanings
| Column Name                                                                 | Description                                                                                                               |
| --------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------- |
| `root_630`                                                                  | **Neuron ID** in FlyWire version 630. Use this to view the neuron at [https://flywire.ai](https://flywire.ai).            |
| `root_783`                                                                  | Neuron ID in FlyWire version 783. Used internally for improved morphology and synapse annotation. Not publicly browsable. |
| `pos_x`, `pos_y`, `pos_z`                                                   | Approximate soma coordinates (in nm) for the neuron within the EM volume.                                                 |
| `nucleus_id`                                                                | ID of the nucleus associated with the neuron (may be `NaN` if unknown).                                                   |
| `side`                                                                      | Hemisphere the neuron resides in: `left` or `right`.                                                                      |
| `ito_lee_hemilineage`                                                       | Hemilineage name based on the Ito & Lee lineage naming convention.                                                        |
| `hartenstein_hemilineage`                                                   | Hemilineage name based on the Hartenstein lineage schema.                                                                 |
| `morphology_group`                                                          | Cluster of neurons with similar morphology, often indicating a cell type.                                                 |
| `cell_class`                                                                | Broad functional group (e.g. `sensory`, `interneuron`, `projection neuron`).                                              |
| `cell_sub_class`                                                            | Subdivision of cell class, if applicable.                                                                                 |
| `cell_type`                                                                 | Named or inferred cell type label (if available).                                                                         |
| `hemibrain_type`                                                            | Matching cell type from the Hemibrain dataset, if available.                                                              |
| `pre`                                                                       | Total number of **presynaptic sites** assigned to this neuron.                                                            |
| `conf_nt`                                                                   | Predicted neurotransmitter with **highest confidence**.                                                                   |
| `conf_nt_p`                                                                 | Proportion of presynaptic sites supporting `conf_nt`. (Confidence value, 0–1).                                            |
| `top_nt`                                                                    | Predicted transmitter based on **top probability**, even if not majority.                                                 |
| `top_nt_p`                                                                  | Probability of `top_nt` according to classifier.                                                                          |
| `known_nt`                                                                  | Literature-based known transmitter, if available.                                                                         |
| `known_nt_source`                                                           | Citation or source for `known_nt`.                                                                                        |
| `acetylcholine`, `gaba`, `glutamate`, `dopamine`, `serotonin`, `octopamine` | Per-class classifier confidence scores for each neurotransmitter (range: 0–1).                                            |
| `segregation_index`                                                         | Measure of how spatially segregated the axon and dendrite compartments are. Higher values imply clearer separation.       |
| `projection_score`                                                          | A score based on axon projection length and morphology—indicative of long-range projection neurons.                       |
| `in_ground_truth`                                                           | `True` if this neuron was used as part of the supervised ground-truth training set.                                       |
| `notes`                                                                     | Free-form field for manual annotations, often empty.                                                                      |

So the key columns I will use are:
* `root_630`: As the NeuronId
* `pos_x`, `pos_y`, and `pos_z`: For the approximate location of the neuron
* `cell_class`
* `cell_type`
* `pre`: The number of pre-synaptic sites.
* `conf_nt`: The models prediction for the neuron.
* `known_nt`: The known neurotransmitter types.
* `acetylcholine`, `gaba`, `glutamate`, `dopamine`, `serotonin`, `octopamine`: The ratio of synapses expressing the neurotransmitter in the neuron.
* `in_ground_truth`: Whether this is in the ground truth dataset or not.


In [ ]:
known_nts = s4_df['known_nt'].value_counts()
display(known_nts)

Ok so there are 72 different mixes here which we will need to trawl through. But first let's only investigate those part of the ground truth 

In [ ]:
df = s4_df[s4_df['in_ground_truth']]
df

Ok just down to 670 rows which is much easier to investigate. Let's find out what NT types are available.

In [ ]:
df['known_nt'].value_counts()

Interesting there are some random ones here. e.g. sparkly, and space blanket. Let's split this up to each class.

In [ ]:
classes = df['known_nt'].unique()
nts_present = []

for cls in classes:
    #split by commas
    cls = str(cls)
    nts = cls.split(',')
    for nt in nts:
        #strip whitespace at the start and end
        nt = nt.strip()
        if (nt not in nts_present) and (nt not in ['nan', 'space blanket', 'sparkly']):
            nts_present.append(nt)
print(f"NTs present in the ground truth: {len(nts_present)}")
print(nts_present)  
    

## Make a clean dataframe for each neuron

In [ ]:
final_df = df[['root_630','pre','pos_x','pos_y','pos_z','cell_class','cell_type','known_nt']]

# now start with one hot encoding the known_nt column
final_df.loc[:,nts_present] = False

for index, row in final_df.iterrows():
    nts = str(row['known_nt']).split(',')
    for nt in nts:
        nt = nt.strip()
        if nt in nts_present:
            final_df.at[index, nt] = True

final_df['num_nts'] = final_df[nts_present].sum(axis=1)
final_df['is_cotransmitter'] = final_df['num_nts'] > 1
final_df
#temp = ['acetylcholine_present','gaba_present','dopamine_present','nitric_oxide_present','Nplp1_present','sNPF_present','amnesiac', 'CG34136', 'CG43117', 'ion-transport peptide', 'Dh44', 'allatostatin-c', 'glutamate', 'CCHa1', 'bursicon', 'proctolin', 'ecdysis- triggering hormone', 'Dh31', 'Ms', 'tachykinin', 'serotonin', 'natalisin']
                                 

# Visualisations

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12, 4))

# Plot the number of cotransmitting neurons
sns.countplot(data=final_df, x='num_nts', ax=ax[0])
ax[0].set_title('Number of Cotransmitting Neurons')
ax[0].set_xlabel('Number of Neurotransmitters')
ax[0].set_ylabel('Count')

# Plot the number of pre-synaptic boutons per neuron
sns.histplot(data=final_df, x='pre', hue='is_cotransmitter', binwidth=250, kde=True, ax=ax[1])
ax[1].set_title('Number of Pre-synaptic Boutons per Neuron')
ax[1].set_xlabel('Number of Pre-synaptic Boutons')
ax[1].set_ylabel('Count')

# Plot the ratios for each neurotransmitter type
temp = {}
for nt in nts_present:
    temp[nt] = final_df[nt].value_counts().to_dict()
temp_df = pd.DataFrame(temp).T

plt.figure(figsize=(12,6))
sns.barplot(x=temp_df.index, y=temp_df[True])
plt.xticks(rotation=90)
plt.xlabel('Neurotransmitter')
plt.ylabel('Number of Neurons')
plt.title('Number of Neurons per Neurotransmitter')
plt.tight_layout()
plt.show()

# sns.barplot(data=pd.DataFrame(temp), ax=ax[1,0])
# ax[1,0].set_title('Neurotransmitter Ratios')
# ax[1,0].set_xlabel('Neurotransmitter')
# ax[1,0].set_ylabel('Count')

So there are lots of what could be better described as neuropeptides here given that the CNN doesn't look for them and only 
* acetylcholine
* gaba
* glutamate
* dopamine
* serotonin
* octopamine

Therefore let's filter to only include these in co-transmission.

In [ ]:
simplified_final_df = final_df[['root_630', 'pre', 'pos_x', 'pos_y', 'pos_z', 'cell_class', 'cell_type', 'known_nt', 'acetylcholine', 'gaba', 'glutamate', 'dopamine', 'serotonin']]
simplified_final_df.loc[:,'octopamine'] = False
simplified_final_df['num_nts'] = simplified_final_df[['acetylcholine', 'gaba', 'glutamate', 'dopamine', 'serotonin', 'octopamine']].sum(axis=1)
simplified_final_df['is_cotransmitter'] = simplified_final_df['num_nts'] > 1
simplified_final_df


In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12, 4))

# Plot the number of cotransmitting neurons
sns.countplot(data=simplified_final_df, x='num_nts', ax=ax[0])
ax[0].set_title('Number of Cotransmitting Neurons')
ax[0].set_xlabel('Number of Neurotransmitters')
ax[0].set_ylabel('Count')

# Plot the number of pre-synaptic boutons per neuron
sns.histplot(data=simplified_final_df, x='pre', hue='is_cotransmitter', binwidth=250, kde=True, ax=ax[1])
ax[1].set_title('Number of Pre-synaptic Boutons per Neuron')
ax[1].set_xlabel('Number of Pre-synaptic Boutons')
ax[1].set_ylabel('Count')

# Plot the ratios for each neurotransmitter type
temp = {}
for nt in ['acetylcholine', 'gaba', 'glutamate', 'dopamine', 'serotonin', 'octopamine']:
    temp[nt] = simplified_final_df[nt].value_counts().to_dict()
temp_df = pd.DataFrame(temp).T

plt.figure(figsize=(12,6))
sns.barplot(x=temp_df.index, y=temp_df[True])
plt.xticks(rotation=90)
plt.xlabel('Neurotransmitter')
plt.ylabel('Number of Neurons')
plt.title('Number of Neurons per Neurotransmitter')
plt.tight_layout()
plt.show()

# Plot the ratios where the neuron is cotransmitting
plt.figure(figsize=(12,6))
# Prepare data for stacked bar plot grouped by is_cotransmitter
nt_cols = ['acetylcholine', 'gaba', 'glutamate', 'dopamine', 'serotonin', 'octopamine']
grouped = simplified_final_df.groupby('is_cotransmitter')[nt_cols].sum().T

# Plot
grouped.plot(kind='bar', stacked=True, figsize=(12,6))
plt.xlabel('Neurotransmitter')
plt.ylabel('Number of Neurons')
plt.title('Number of Neurons per Neurotransmitter (Grouped by Co-transmission)')
plt.xticks(rotation=90)
plt.legend(title='is_cotransmitter', labels=['Single NT', 'Co-transmitter'])
plt.tight_layout()
plt.show()

In [ ]:
simplified_final_df['pre'].describe()

In [ ]:
# Save the final DataFrame to a CSV file
simplified_final_df.to_csv('../data/simplified_DataS4.csv', index=False)